<a href="https://colab.research.google.com/github/januvishwanath56-debug/Gen-AI-Assignments/blob/main/Assignment-3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 ASSIGNMENT NLP – 3: Build a Chatbot using Hugging Face Transformers
## Data Science Internship – February 2026 | Innomatics Research Labs

---

| Detail | Info |
|---|---|
| **Task** | Build a Chatbot using Hugging Face Transformers |
| **Model Used** | `microsoft/DialoGPT-medium` |
| **Framework** | PyTorch + Hugging Face Transformers |
| **Interface** | Console-based Interactive Chatbot |
| **Deadline** | 01/04/2026 |

---

## 📌 Objective
Build a simple conversational chatbot using a **pre-trained transformer model** from Hugging Face that can interact with users and generate meaningful, dynamic responses — without any hardcoded answers.

## 🎯 Learning Outcomes
After completing this assignment, we will understand:
- How **transformer-based language models** work internally
- How to use **pre-trained NLP models** from the Hugging Face Model Hub
- How to perform **text generation** using transformer models
- How to build an **interactive conversational system**
- How **prompt-based text generation** works

---

## 📦 Section 1: Install Required Libraries

We install:
- **`transformers`** — Hugging Face library for pre-trained NLP models
- **`torch`** — PyTorch backend for model inference
- **`sentencepiece`** — Tokenization dependency for some models

> ⚠️ **Note:** Restart the runtime after running this cell if you're on Google Colab.

In [1]:
# Install all required libraries
!pip install transformers torch sentencepiece --quiet

print("✅ All libraries installed successfully!")

✅ All libraries installed successfully!


## 📚 Section 2: Import Libraries & Verify Setup

### Key Imports:
| Import | Purpose |
|---|---|
| `AutoModelForCausalLM` | Loads a causal (generative) language model |
| `AutoTokenizer` | Converts text ↔ token IDs |
| `torch` | PyTorch tensor operations & device management |

In [2]:
# ── Core imports ──────────────────────────────────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# ── Verify setup ──────────────────────────────────────────────────────────────
print("✅ Libraries imported successfully!")
print(f"   PyTorch version  : {torch.__version__}")

# Detect available device — GPU speeds up inference significantly
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"   Device selected  : {DEVICE.upper()}")

if DEVICE == "cuda":
    print(f"   GPU Name         : {torch.cuda.get_device_name(0)}")
else:
    print("   ℹ️  No GPU found — running on CPU (slightly slower, still works fine)")

✅ Libraries imported successfully!
   PyTorch version  : 2.10.0+cu128
   Device selected  : CUDA
   GPU Name         : Tesla T4


## 🧠 Section 3: Load Pre-trained Model and Tokenizer

### Why DialoGPT?
**DialoGPT** (by Microsoft) is a large-scale pre-trained dialogue response generation model trained on **147M Reddit conversation threads**. It is purpose-built for multi-turn conversational tasks.

### Model Variants:
| Variant | Parameters | Memory | Use Case |
|---|---|---|---|
| `DialoGPT-small` | ~117M | ~500MB | Fast, low memory |
| `DialoGPT-medium` | ~345M | ~1.5GB | **Best balance ✅** |
| `DialoGPT-large` | ~762M | ~3GB | Highest quality, needs GPU |

### How the Tokenizer Works:
```
Raw Text  →  [Token IDs]  →  Model  →  [Token IDs]  →  Decoded Text
"Hello"   →  [15496]     →  ...    →  [986, 423]   →  "How are you?"
```

In [3]:
# ── Model configuration ───────────────────────────────────────────────────────
MODEL_NAME = "microsoft/DialoGPT-medium"   # Pre-trained conversational model

print(f"📥 Loading model: {MODEL_NAME}")
print("   This may take a moment on first run (downloads ~1.5 GB model weights)...\n")

# ── Load Tokenizer ────────────────────────────────────────────────────────────
# The tokenizer converts human-readable text into numerical token IDs
# and decodes model output token IDs back into text
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token   # Required: set pad token = end-of-sequence token
print("   ✅ Tokenizer loaded")
print(f"      Vocabulary size      : {tokenizer.vocab_size:,} tokens")
print(f"      EOS token            : '{tokenizer.eos_token}' (ID: {tokenizer.eos_token_id})")

# ── Load Model ────────────────────────────────────────────────────────────────
# AutoModelForCausalLM is used for causal (left-to-right) text generation
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)   # Move model to GPU if available
model.eval()               # Set to evaluation mode (disables dropout, etc.)

print("   ✅ Model loaded and ready")
print(f"      Model device         : {next(model.parameters()).device}")
total_params = sum(p.numel() for p in model.parameters())
print(f"      Total parameters     : {total_params:,} (~{total_params/1e6:.0f}M)")
print("\n🎉 Model and tokenizer are ready for conversation!")

📥 Loading model: microsoft/DialoGPT-medium
   This may take a moment on first run (downloads ~1.5 GB model weights)...



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

   ✅ Model loaded and ready
      Model device         : cuda:0
      Total parameters     : 406,286,336 (~406M)

🎉 Model and tokenizer are ready for conversation!


## ⚙️ Section 4: Text Generation — Core Function

### How Response Generation Works:

```
Step 1: Encode user input → token IDs
Step 2: Append EOS token  → signals end of user's turn
Step 3: Concatenate with  → full conversation history
Step 4: Feed to model     → model predicts next tokens
Step 5: Decode output     → convert token IDs back to text
Step 6: Extract reply     → isolate only the new response
```

### Key Generation Parameters:
| Parameter | Value | Effect |
|---|---|---|
| `max_new_tokens` | 150 | Max length of generated response |
| `temperature` | 0.7 | Lower = more focused, Higher = more creative |
| `top_p` | 0.9 | Nucleus sampling — keeps top 90% probability mass |
| `top_k` | 50 | Considers only top 50 most likely tokens |
| `do_sample` | True | Enables probabilistic sampling (non-greedy) |
| `repetition_penalty` | 1.3 | Discourages repeating the same phrases |

In [4]:
def generate_response(user_input, chat_history_ids=None, max_history_turns=5):
    """
    Generate a chatbot response using DialoGPT.

    Parameters:
    -----------
    user_input       : str   — The user's latest message
    chat_history_ids : Tensor — Encoded previous conversation turns (None for first turn)
    max_history_turns: int   — Max past turns to keep (avoids exceeding context window)

    Returns:
    --------
    response         : str   — The model's generated reply
    new_history_ids  : Tensor — Updated conversation history (for next turn)
    """

    # Step 1: Encode the user's input text → token IDs
    # Append EOS token (eos_token_id) to signal end of user's turn to the model
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"          # Return as PyTorch tensor
    ).to(DEVICE)

    # Step 2: Build the full input by appending new message to conversation history
    if chat_history_ids is not None:
        # Truncate history if it gets too long (keep last N turns to stay within model limits)
        max_history_length = 800   # Keep history manageable
        if chat_history_ids.shape[-1] > max_history_length:
            chat_history_ids = chat_history_ids[:, -max_history_length:]

        # Concatenate history + new user message along the sequence dimension
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        # First turn: no history yet
        bot_input_ids = new_input_ids

    # Step 3: Generate the model's response
    # torch.no_grad() disables gradient computation → saves memory during inference
    with torch.no_grad():
        output_ids = model.generate(
            bot_input_ids,
            max_new_tokens=150,         # Max tokens to generate in response
            do_sample=True,             # Use sampling (not greedy) for varied responses
            temperature=0.7,            # Controls randomness: lower = more focused
            top_p=0.9,                  # Nucleus sampling: consider top 90% probability mass
            top_k=50,                   # Only consider top 50 candidate tokens at each step
            repetition_penalty=1.3,     # Penalise repeated phrases in output
            pad_token_id=tokenizer.eos_token_id,  # Use EOS as padding token
            eos_token_id=tokenizer.eos_token_id   # Stop generation at EOS
        )

    # Step 4: Extract ONLY the new response tokens (skip the input tokens)
    # output_ids includes the full input + generated tokens
    response_ids = output_ids[:, bot_input_ids.shape[-1]:]

    # Step 5: Decode token IDs → human-readable text
    response = tokenizer.decode(
        response_ids[0],
        skip_special_tokens=True    # Remove EOS, PAD, and other special tokens
    ).strip()

    # Step 6: Return response and updated history (full output_ids for next turn)
    return response, output_ids


print("✅ generate_response() function defined successfully!")
print("   The function handles: encoding → history → generation → decoding")

✅ generate_response() function defined successfully!
   The function handles: encoding → history → generation → decoding


## 💬 Section 5: Chatbot Interaction Demo (Simulated)

Before launching the full interactive loop, let's run a **simulated demo** with predefined inputs to demonstrate that the pipeline works correctly and to capture outputs for the notebook submission.

> 📝 This section fulfils the **"Show chatbot interaction outputs"** submission requirement.

In [5]:
# ── Simulated Conversation Demo ───────────────────────────────────────────────
# Predefined inputs to demonstrate chatbot behaviour in a visible, saved output

demo_inputs = [
    "Hello",
    "What is Artificial Intelligence?",
    "Who created Python?",
    "What is Natural Language Processing?",
    "Can you tell me about transformers in NLP?",
    "Thank you"
]

print("=" * 65)
print("          🤖  DialoGPT Chatbot — DEMO SESSION")
print("=" * 65)
print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
print("-" * 65)

# Track conversation history across turns
demo_history = None

for user_msg in demo_inputs:
    print(f"\nUser    : {user_msg}")

    # Generate response using the model
    bot_reply, demo_history = generate_response(user_msg, demo_history)

    # Fallback: if model returns empty output, use a polite default
    if not bot_reply:
        bot_reply = "I'm not sure about that, but I'm here to help! Could you rephrase?"

    print(f"Chatbot : {bot_reply}")
    print("-" * 65)

print("\nChatbot : It was great talking with you! Goodbye! 👋")
print("=" * 65)
print("\n✅ Demo session complete — pipeline working correctly!")

          🤖  DialoGPT Chatbot — DEMO SESSION
Chatbot: Hello! I am your AI assistant. How can I help you today?
-----------------------------------------------------------------

User    : Hello


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Chatbot : Hey! Nice to meet you.
-----------------------------------------------------------------

User    : What is Artificial Intelligence?
Chatbot : I'm not sure but I think it's a computer program that uses our brain and creates an artificial intelligence.
-----------------------------------------------------------------

User    : Who created Python?
Chatbot : The creator of Python, Python was first made in the early 70s by Richard Dawkins, a biologist who used his own creation as inspiration for programming languages.
-----------------------------------------------------------------

User    : What is Natural Language Processing?
Chatbot : That one is my favorite too :D
-----------------------------------------------------------------

User    : Can you tell me about transformers in NLP?
Chatbot : It's a machine learning algorithm?
-----------------------------------------------------------------

User    : Thank you
Chatbot : Ohhhh, right. Thanks :P
----------------------------

## 🔄 Section 6: Full Interactive Chatbot (Console Loop)

This is the **main interactive chatbot** that:
1. Accepts real user input from the console
2. Generates responses dynamically using the transformer model
3. Maintains full conversation history across turns
4. Exits cleanly when the user types **`exit`** or **`quit`**

### Expected Pipeline Flow:
```
User Input → Tokenize → Append History → Model.generate() → Decode → Display → Loop
                                                                               ↓
                                                                   (until 'exit'/'quit')
```

> ⚠️ **Note for Jupyter/Colab users:** This cell uses `input()` for real-time interaction. Run it and type your messages in the input box that appears below the cell. Type `exit` to stop.

In [10]:
def run_chatbot():
    """
    Main chatbot loop.
    - Continuously accepts user input
    - Generates model-based responses (no hardcoded answers)
    - Maintains multi-turn conversation history
    - Exits on 'exit' or 'quit'
    """

    print("=" * 65)
    print("         🤖  DialoGPT Chatbot — INTERACTIVE SESSION")
    print("=" * 65)
    print("  Powered by: microsoft/DialoGPT-medium (Hugging Face)")
    print("  Type 'exit' or 'quit' to end the conversation.")
    print("=" * 65)
    print("\nChatbot: Hello! I am your AI assistant. How can I help you today?\n")

    # Initialise conversation history as None (first turn has no history)
    chat_history_ids = None
    turn_number = 0

    # ── Main conversation loop ─────────────────────────────────────────────────
    while True:
        # Step 1: Accept user input from console
        try:
            user_input = input("You     : ").strip()
        except (EOFError, KeyboardInterrupt):
            # Handle Ctrl+C or end-of-input gracefully
            print("\n\nChatbot : Session interrupted. Goodbye! 👋")
            break

        # Step 2: Skip blank inputs — ask user to type something
        if not user_input:
            print("Chatbot : Please type a message to continue.\n")
            continue

        # Step 3: Check for exit keywords (case-insensitive)
        if user_input.lower() in ["exit", "quit", "bye", "goodbye"]:
            print("\nChatbot : Thank you for chatting with me! Have a great day! 👋")
            print(f"          (Total conversation turns: {turn_number})")
            print("=" * 65)
            break

        # Step 4: Generate a response using the transformer model
        # This is where the Hugging Face model does the heavy lifting
        print("Chatbot : ", end="", flush=True)   # Print prefix before response appears

        try:
            response, chat_history_ids = generate_response(user_input, chat_history_ids)

            # Fallback if model returns an empty string
            if not response:
                response = "That's interesting! Could you tell me more?"

        except Exception as e:
            # Gracefully handle any unexpected model errors
            response = "I encountered an issue generating a response. Could you rephrase?"
            print(f"  [Debug] Error: {e}")

        # Step 5: Display the chatbot's response
        print(response)
        print()   # Blank line for readability

        # Increment turn counter
        turn_number += 1


# ── Launch the chatbot ────────────────────────────────────────────────────────
run_chatbot()

         🤖  DialoGPT Chatbot — INTERACTIVE SESSION
  Powered by: microsoft/DialoGPT-medium (Hugging Face)
  Type 'exit' or 'quit' to end the conversation.

Chatbot: Hello! I am your AI assistant. How can I help you today?

You     : What is Natural Language Processing?
Chatbot : The software that gives you a machine readable language. I'm guessing it's for writing code, not programming?

You     : Who created Python?
Chatbot : I love this thread!



Chatbot : Session interrupted. Goodbye! 👋


## 📊 Section 7: Understanding the Model — Concept Summary

This section explains the key NLP and transformer concepts used in this assignment.

In [7]:
# ── Concept Exploration: Tokenization ─────────────────────────────────────────
# Demonstrates how the tokenizer converts text to token IDs and back

sample_text = "What is Natural Language Processing?"

# Encode: text → token IDs
token_ids = tokenizer.encode(sample_text)

# Decode individual tokens to see what each token represents
tokens = [tokenizer.decode([tid]) for tid in token_ids]

print("🔤 TOKENIZATION DEMO")
print("=" * 55)
print(f"Input Text   : {sample_text}")
print(f"Token IDs    : {token_ids}")
print(f"Tokens       : {tokens}")
print(f"Token Count  : {len(token_ids)} tokens")
print(f"\nDecoded back : {tokenizer.decode(token_ids)}")
print("=" * 55)
print("\n💡 Key Insight: The model never sees raw text — it only sees")
print("   integer token IDs, which it maps to learned embeddings.")

🔤 TOKENIZATION DEMO
Input Text   : What is Natural Language Processing?
Token IDs    : [2061, 318, 12068, 15417, 28403, 30]
Tokens       : ['What', ' is', ' Natural', ' Language', ' Processing', '?']
Token Count  : 6 tokens

Decoded back : What is Natural Language Processing?

💡 Key Insight: The model never sees raw text — it only sees
   integer token IDs, which it maps to learned embeddings.


In [8]:
# ── Concept Exploration: Generation Parameters Effect ─────────────────────────
# Show how temperature affects response diversity

test_input = "Tell me something interesting"
input_ids = tokenizer.encode(
    test_input + tokenizer.eos_token, return_tensors="pt"
).to(DEVICE)

print("🌡️  TEMPERATURE EFFECT DEMO")
print("=" * 55)
print(f"Input: '{test_input}'\n")

temperatures = [
    (0.3, "Low temperature  → Focused/Predictable"),
    (0.7, "Mid temperature  → Balanced (Used in chatbot)"),
    (1.2, "High temperature → Creative/Diverse")
]

with torch.no_grad():
    for temp, label in temperatures:
        out = model.generate(
            input_ids,
            max_new_tokens=40,
            do_sample=True,
            temperature=temp,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        reply = tokenizer.decode(
            out[:, input_ids.shape[-1]:][0], skip_special_tokens=True
        ).strip()
        print(f"  Temp={temp} | {label}")
        print(f"  Response: {reply if reply else '(empty — model chose EOS early)'}")
        print()

print("=" * 55)
print("💡 Higher temperature = more creative but less coherent.")
print("   Lower temperature = more predictable but safer responses.")

🌡️  TEMPERATURE EFFECT DEMO
Input: 'Tell me something interesting'

  Temp=0.3 | Low temperature  → Focused/Predictable
  Response: I'm not sure what you mean by interesting.

  Temp=0.7 | Mid temperature  → Balanced (Used in chatbot)
  Response: I don't know, but I'm sure there's a lot of interesting things you could tell.

  Temp=1.2 | High temperature → Creative/Diverse
  Response: Just the usual stuff

💡 Higher temperature = more creative but less coherent.
   Lower temperature = more predictable but safer responses.


## 📝 Section 8: Key Learnings & Conclusion

### 🔑 Key Concepts Learned

| # | Concept | Description |
|---|---|---|
| 1 | **Transformer Architecture** | Self-attention mechanism that processes all tokens in parallel, capturing long-range dependencies |
| 2 | **Tokenization** | Converting raw text into numerical token IDs that the model can process |
| 3 | **Pre-trained Models** | Models like DialoGPT trained on massive datasets — fine-tuned on dialogue data |
| 4 | **Causal Language Modelling** | Generating text left-to-right by predicting the next token given all previous tokens |
| 5 | **Sampling Strategies** | `temperature`, `top_p`, `top_k` control randomness and diversity of generated text |
| 6 | **Conversation History** | Concatenating past turns into context so the model generates coherent multi-turn replies |
| 7 | **Hugging Face Hub** | Central repository for thousands of pre-trained NLP models accessible via simple APIs |

### ✅ Assignment Checklist
- [x] Pre-trained Hugging Face model loaded (`DialoGPT-medium`)
- [x] No hardcoded responses — all replies are model-generated
- [x] Multi-turn conversation history maintained
- [x] Exit condition implemented (`exit` / `quit`)
- [x] Simulated demo with visible outputs
- [x] Interactive console loop
- [x] Clean, well-commented code
- [x] Key concepts explained with examples

### 🚀 Possible Extensions
- Replace DialoGPT with **GPT-2**, **BlenderBot**, or **LLaMA** for different behaviours
- Add a **Gradio** or **Streamlit** web UI for a browser-based chat interface
- Fine-tune the model on a **custom domain-specific dataset**
- Integrate **sentiment analysis** to detect user emotion and adjust tone
- Add **memory summarisation** to handle very long conversations

In [9]:
# ── Final Summary ─────────────────────────────────────────────────────────────
print("=" * 65)
print("   ✅ ASSIGNMENT NLP-3 COMPLETE")
print("=" * 65)
print(f"   Model          : microsoft/DialoGPT-medium")
print(f"   Framework      : Hugging Face Transformers + PyTorch")
print(f"   Device used    : {DEVICE.upper()}")
print(f"   Vocab size     : {tokenizer.vocab_size:,} tokens")
print(f"   Parameters     : ~345M")
print("")
print("   Key NLP Concepts Demonstrated:")
concepts = [
    "Transformer Architecture",
    "Tokenization & Encoding",
    "Pre-trained Language Models",
    "Causal Language Modelling",
    "Sampling Strategies (temperature, top_p, top_k)",
    "Multi-turn Conversation History",
    "Hugging Face Model Hub"
]
for i, concept in enumerate(concepts, 1):
    print(f"   {i}. {concept}")
print("=" * 65)
print("   Submitted for: Innomatics Research Labs — Feb 2026 Internship")
print("=" * 65)

   ✅ ASSIGNMENT NLP-3 COMPLETE
   Model          : microsoft/DialoGPT-medium
   Framework      : Hugging Face Transformers + PyTorch
   Device used    : CUDA
   Vocab size     : 50,257 tokens
   Parameters     : ~345M

   Key NLP Concepts Demonstrated:
   1. Transformer Architecture
   2. Tokenization & Encoding
   3. Pre-trained Language Models
   4. Causal Language Modelling
   5. Sampling Strategies (temperature, top_p, top_k)
   6. Multi-turn Conversation History
   7. Hugging Face Model Hub
   Submitted for: Innomatics Research Labs — Feb 2026 Internship
